# Project 3 Week 2: 3D Rigid-Body Air-Jet Sorting Simulator

사용 방법:
1. 위에서부터 모든 셀을 순서대로 실행합니다.
2. 마지막 Simulation 셀까지 실행하면 interactive widget이 나타납니다.
3. `Update plot` 또는 `Animate 3D` 버튼으로 결과를 확인합니다.
4. `Animate 3D` 버튼 클릭 후 애니메이션 창이 켜졌다가 사라지는 경우, 다시 한 번 눌러주세요.

# 0. Imports and global settings


In [ ]:
# ============================================================
# Project 3 Week 2
# 3D Rigid-Body Air-Jet Sorting Simulator
# Revised sorting-criterion version
#
# Major conceptual update:
# - x: conveyor direction
# - y: conveyor width / lateral input coordinate / jet targeting coordinate
# - z: vertical direction
#
# Sorting success is now based on x landing pass region:
#   pass-class object:
#       success if x_pass_min <= x_land <= x_pass_max
#   reject-class object:
#       success if x_land is outside the x-pass region
#
# Conveyor:
#   fixed x range: -0.2 m to 0.0 m
#
# Jet:
#   adjustable 3D position: x, y, z
#   adjustable direction: azimuth, elevation
#
# Object:
#   plate, rod, cylinder, sphere, irregular flake
#   adjustable dimensions and surface resolution
#
# 필요시 한 번만 실행:
#   %pip install ipywidgets
# ============================================================

import math
import warnings
from dataclasses import dataclass
from typing import Tuple

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

warnings.filterwarnings("ignore")
plt.rcParams["animation.embed_limit"] = 200

# 1. Quaternion utilities

In [ ]:
# ============================================================
# 1. Quaternion utilities
# ============================================================

def quat_normalize(q):
    q = np.array(q, dtype=float)
    n = np.linalg.norm(q)
    if n < 1e-15:
        return np.array([1.0, 0.0, 0.0, 0.0])
    return q / n


def quat_multiply(q1, q2):
    """
    Quaternion product.
    Convention: q = [w, x, y, z]
    """
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2

    return np.array([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2
    ], dtype=float)


def quat_to_rotmat(q):
    """
    Convert quaternion to rotation matrix.
    q maps body-frame vector to world-frame vector:
        p_world = R(q) p_body
    """
    q = quat_normalize(q)
    w, x, y, z = q

    return np.array([
        [1 - 2*(y*y + z*z),     2*(x*y - z*w),     2*(x*z + y*w)],
        [    2*(x*y + z*w), 1 - 2*(x*x + z*z),     2*(y*z - x*w)],
        [    2*(x*z - y*w),     2*(y*z + x*w), 1 - 2*(x*x + y*y)]
    ], dtype=float)


def quat_from_euler_deg(roll_deg=0.0, pitch_deg=0.0, yaw_deg=0.0):
    """
    Euler angle to quaternion.
    roll: rotation about x
    pitch: rotation about y
    yaw: rotation about z
    """
    roll = np.deg2rad(roll_deg)
    pitch = np.deg2rad(pitch_deg)
    yaw = np.deg2rad(yaw_deg)

    cr, sr = np.cos(roll/2), np.sin(roll/2)
    cp, sp = np.cos(pitch/2), np.sin(pitch/2)
    cy, sy = np.cos(yaw/2), np.sin(yaw/2)

    q = np.array([
        cr*cp*cy + sr*sp*sy,
        sr*cp*cy - cr*sp*sy,
        cr*sp*cy + sr*cp*sy,
        cr*cp*sy - sr*sp*cy
    ], dtype=float)

    return quat_normalize(q)


def quat_to_euler_deg(q):
    q = quat_normalize(q)
    w, x, y, z = q

    sinr_cosp = 2 * (w*x + y*z)
    cosr_cosp = 1 - 2 * (x*x + y*y)
    roll = np.arctan2(sinr_cosp, cosr_cosp)

    sinp = 2 * (w*y - z*x)
    sinp = np.clip(sinp, -1.0, 1.0)
    pitch = np.arcsin(sinp)

    siny_cosp = 2 * (w*z + x*y)
    cosy_cosp = 1 - 2 * (y*y + z*z)
    yaw = np.arctan2(siny_cosp, cosy_cosp)

    return np.rad2deg([roll, pitch, yaw])


def update_quaternion(q, omega_world, dt):
    """
    World-angular-velocity form:
        dq/dt = 0.5 * (0, omega_world) ⊗ q
    """
    omega_quat = np.array([0.0, omega_world[0], omega_world[1], omega_world[2]])
    q_dot = 0.5 * quat_multiply(omega_quat, q)
    return quat_normalize(q + dt * q_dot)


# 2. Data classes


In [ ]:
# ============================================================
# 2. Data classes
# ============================================================

@dataclass
class RigidObject:
    name: str
    mass: float
    cd: float
    points_body: np.ndarray
    area_weights: np.ndarray
    inertia_body: np.ndarray
    half_extents: np.ndarray
    area_ref: float


@dataclass
class ConveyorGeometry:
    """
    Fixed conveyor:
        x_start = -0.2 m
        x_end   =  0.0 m

    Object leaves the conveyor when x >= x_end,
    unless it detaches earlier by impulse.
    """
    x_start: float = -0.20
    x_end: float = 0.00
    width: float = 0.60
    height: float = 0.50
    landing_z: float = 0.0


@dataclass
class JetField:
    """
    3D Gaussian jet.

    Jet center:
        r_j = (x_center, y_center, z_center)

    Jet direction:
        azimuth/elevation.

    azimuth_deg:
        0 deg: +x
        +90 deg: +y
        -90 deg: -y
        180 deg or -180 deg: -x
        
    elevation_deg:
        0 deg: horizontal
        positive: +z component
        negative: -z component

    Gaussian profile:
        u_jet = U0 exp(-d_perp^2/(2 sigma^2)) e_jet
    """
    Umax: float = 16.0
    sigma: float = 0.055
    t_on: float = 0.18
    duration: float = 0.18

    x_center: float = -0.08
    y_center: float = 0.00
    z_center: float = 0.50

    azimuth_deg: float = 0.0
    elevation_deg: float = 0.0

    noise_std: float = 0.00

    def time_active(self, t):
        return self.t_on <= t <= self.t_on + self.duration

    def direction_unit(self):
        """
        New azimuth convention:
    
        azimuth = 0 deg    -> +x direction, conveyor direction
        azimuth = 90 deg   -> +y direction
        azimuth = -90 deg  -> -y direction
        azimuth = 180 deg  -> -x direction
    
        elevation = 0 deg  -> horizontal jet
        elevation > 0 deg  -> upward +z component
        elevation < 0 deg  -> downward -z component
        """
        az = np.deg2rad(self.azimuth_deg)
        el = np.deg2rad(self.elevation_deg)
    
        ex = np.cos(az) * np.cos(el)
        ey = np.sin(az) * np.cos(el)
        ez = np.sin(el)
    
        e = np.array([ex, ey, ez], dtype=float)
        n = np.linalg.norm(e)
    
        if n < 1e-15:
            return np.array([1.0, 0.0, 0.0])
    
        return e / n
    def center_vector(self):
        return np.array([self.x_center, self.y_center, self.z_center], dtype=float)


@dataclass
class SimulationConfig3D:
    rho_air: float = 1.225
    g: float = 9.81
    dt: float = 0.002
    t_max: float = 2.0

    conveyor_speed: float = 1.20
    J_detach: float = 0.003

    x_pass_min: float = 0.20
    x_pass_max: float = 0.55
    object_should_pass: bool = True

    seed: int = 1


@dataclass
class InitialCondition3D:
    x0: float = -0.20
    y0: float = 0.00
    z0: float = 0.50
    q0: np.ndarray = None
    omega0: np.ndarray = None


@dataclass
class SimulationResult3D:
    data: pd.DataFrame
    obj: RigidObject
    jet: JetField
    geom: ConveyorGeometry
    config: SimulationConfig3D
    initial: InitialCondition3D

    @property
    def landing_x(self):
        return float(self.data["x"].iloc[-1])

    @property
    def landing_y(self):
        return float(self.data["y"].iloc[-1])

    @property
    def landing_z(self):
        return float(self.data["z"].iloc[-1])

    @property
    def flight_time(self):
        return float(self.data["t"].iloc[-1])

    @property
    def detached(self):
        return bool((self.data["phase"] == "flight").any() or (self.data["phase"] == "landed").any())

    @property
    def detachment_time(self):
        rows = self.data[self.data["detached"] == True]
        if len(rows) == 0:
            return np.nan
        return float(rows["t"].iloc[0])

    @property
    def detachment_x(self):
        rows = self.data[self.data["detached"] == True]
        if len(rows) == 0:
            return np.nan
        return float(rows["x"].iloc[0])

    @property
    def J_vec(self):
        return self.data[["Jx", "Jy", "Jz"]].iloc[-1].to_numpy(dtype=float)

    @property
    def L_vec(self):
        return self.data[["Lx", "Ly", "Lz"]].iloc[-1].to_numpy(dtype=float)

    @property
    def J_norm(self):
        return float(np.linalg.norm(self.J_vec))

    @property
    def L_norm(self):
        return float(np.linalg.norm(self.L_vec))

    @property
    def max_omega(self):
        return float(self.data["omega_norm"].max())

    def in_pass_region(self):
        lo = min(self.config.x_pass_min, self.config.x_pass_max)
        hi = max(self.config.x_pass_min, self.config.x_pass_max)
        return lo <= self.landing_x <= hi

    def sorting_success(self):
        """
        Revised success criterion.

        pass-class object:
            success if x_land is inside pass region

        reject-class object:
            success if x_land is outside pass region
        """
        in_region = self.in_pass_region()

        if self.config.object_should_pass:
            return in_region
        else:
            return not in_region


# 3. Surface point generation and rigid object creation


In [ ]:
# ============================================================
# 3. Surface point generation and rigid object creation
# ============================================================

def inertia_from_points(points_body, mass):
    """
    Approximate inertia tensor from surface representative points.
    """
    pts = np.array(points_body, dtype=float)
    N = len(pts)
    if N == 0:
        return np.eye(3) * 1e-8

    mi = mass / N
    I = np.zeros((3, 3), dtype=float)

    for r in pts:
        r2 = float(np.dot(r, r))
        I += mi * (r2 * np.eye(3) - np.outer(r, r))

    I += np.eye(3) * 1e-8
    return I


def make_cuboid_surface_points(lx, ly, lz, resolution=7):
    """
    Surface points for rectangular cuboid.
    Used for plate and rod-like object.
    """
    r = max(int(resolution), 3)
    nx = max(r, 3)
    ny = max(r, 3)
    nz = max(int(np.ceil(r / 2)), 3)

    xs = np.linspace(-lx/2, lx/2, nx)
    ys = np.linspace(-ly/2, ly/2, ny)
    zs = np.linspace(-lz/2, lz/2, nz)

    pts = []

    for x in xs:
        for y in ys:
            pts.append([x, y, -lz/2])
            pts.append([x, y,  lz/2])

    for x in xs:
        for z in zs:
            pts.append([x, -ly/2, z])
            pts.append([x,  ly/2, z])

    for y in ys:
        for z in zs:
            pts.append([-lx/2, y, z])
            pts.append([ lx/2, y, z])

    pts = np.unique(np.round(np.array(pts, dtype=float), decimals=10), axis=0)
    return pts


def make_cylinder_surface_points(radius, height, resolution=10):
    """
    Cylinder surface points.
    Cylinder axis is body x direction.
    height = cylinder length along body x.
    """
    r = max(int(resolution), 5)
    n_theta = max(2 * r, 12)
    n_x = max(r, 5)
    n_rad = max(int(np.ceil(r / 2)), 3)

    xs = np.linspace(-height/2, height/2, n_x)
    theta = np.linspace(0, 2*np.pi, n_theta, endpoint=False)

    pts = []

    # side surface
    for x in xs:
        for th in theta:
            pts.append([x, radius*np.cos(th), radius*np.sin(th)])

    # end caps
    radial = np.linspace(0, radius, n_rad)
    for xcap in [-height/2, height/2]:
        for rr in radial:
            for th in theta:
                pts.append([xcap, rr*np.cos(th), rr*np.sin(th)])

    pts = np.unique(np.round(np.array(pts, dtype=float), decimals=10), axis=0)
    return pts


def make_sphere_surface_points(radius, resolution=12):
    """
    Sphere surface points.
    """
    r = max(int(resolution), 6)
    n_theta = r
    n_phi = 2 * r

    theta = np.linspace(0, np.pi, n_theta)
    phi = np.linspace(0, 2*np.pi, n_phi, endpoint=False)

    pts = []

    for th in theta:
        for ph in phi:
            x = radius * np.sin(th) * np.cos(ph)
            y = radius * np.sin(th) * np.sin(ph)
            z = radius * np.cos(th)
            pts.append([x, y, z])

    pts = np.unique(np.round(np.array(pts, dtype=float), decimals=10), axis=0)
    return pts


def make_irregular_flake_points(lx, ly, lz, resolution=10, seed=4):
    """
    Irregular flake point cloud.
    lx, ly, lz control approximate size.
    """
    rng = np.random.default_rng(seed)
    n = max(8 * int(resolution), 40)

    angles = rng.uniform(0, 2*np.pi, n)
    radii = rng.uniform(0.2, 1.0, n)

    x = (lx/2) * radii * np.cos(angles) * (1 + 0.25*rng.normal(size=n))
    y = (ly/2) * radii * np.sin(angles) * (1 + 0.35*rng.normal(size=n))
    z = (lz/2) * rng.normal(size=n)

    extra = np.array([
        [ 0.50*lx,  0.25*ly,  0.40*lz],
        [ 0.40*lx, -0.45*ly, -0.30*lz],
        [-0.50*lx,  0.40*ly,  0.20*lz],
        [-0.35*lx, -0.55*ly,  0.50*lz],
        [ 0.10*lx,  0.60*ly, -0.40*lz],
    ])

    pts = np.column_stack([x, y, z])
    pts = np.vstack([pts, extra])
    pts = pts - pts.mean(axis=0, keepdims=True)

    return pts


def create_rigid_object(
    kind="thin plate",
    mass=0.006,
    cd=1.20,
    length_x=0.100,
    width_y=0.065,
    thickness_z=0.008,
    cylinder_radius=0.015,
    cylinder_height=0.120,
    sphere_radius=0.035,
    resolution=8,
    irregular_seed=4
):
    """
    Create adjustable rigid object.

    kind:
        thin plate
        rod-like object
        cylinder
        sphere
        irregular flake
    """
    kind = kind.lower()

    if kind == "thin plate":
        lx = length_x
        ly = width_y
        lz = thickness_z
        pts = make_cuboid_surface_points(lx, ly, lz, resolution=resolution)
        total_area = 2 * (lx*ly + lx*lz + ly*lz)
        name = "thin plate"
        half_extents = np.array([lx/2, ly/2, lz/2])

    elif kind == "rod-like object":
        lx = length_x
        ly = width_y
        lz = thickness_z
        pts = make_cuboid_surface_points(lx, ly, lz, resolution=resolution)
        total_area = 2 * (lx*ly + lx*lz + ly*lz)
        name = "rod-like object"
        half_extents = np.array([lx/2, ly/2, lz/2])

    elif kind == "cylinder":
        r = cylinder_radius
        h = cylinder_height
        pts = make_cylinder_surface_points(r, h, resolution=resolution)
        total_area = 2*np.pi*r*h + 2*np.pi*r*r
        name = "cylinder"
        half_extents = np.array([h/2, r, r])

    elif kind == "sphere":
        r = sphere_radius
        pts = make_sphere_surface_points(r, resolution=resolution)
        total_area = 4*np.pi*r*r
        name = "sphere"
        half_extents = np.array([r, r, r])

    elif kind == "irregular flake":
        lx = length_x
        ly = width_y
        lz = thickness_z
        pts = make_irregular_flake_points(
            lx=lx,
            ly=ly,
            lz=lz,
            resolution=resolution,
            seed=irregular_seed
        )
        total_area = 1.30 * max(lx*ly, 1e-8)
        name = "irregular flake"
        mins = pts.min(axis=0)
        maxs = pts.max(axis=0)
        half_extents = 0.5 * (maxs - mins)

    else:
        raise ValueError("Unknown object kind.")

    if len(pts) < 1:
        raise ValueError("No surface points generated.")

    area_weights = np.ones(len(pts), dtype=float) * total_area / len(pts)
    inertia_body = inertia_from_points(pts, mass)
    area_ref = float(np.sum(area_weights))

    return RigidObject(
        name=name,
        mass=mass,
        cd=cd,
        points_body=pts,
        area_weights=area_weights,
        inertia_body=inertia_body,
        half_extents=half_extents,
        area_ref=area_ref
    )


# 4. Kinematics and force model


In [ ]:
# ============================================================
# 4. Kinematics and force model
# ============================================================

def transform_surface_points(obj, r_com, q):
    R = quat_to_rotmat(q)
    ri_world = obj.points_body @ R.T
    p_world = r_com.reshape(1, 3) + ri_world
    return p_world, ri_world


def gaussian_jet_velocity(points_world, t, jet, rng=None):
    """
    General 3D Gaussian jet field.

    u_jet(r,t) = U0(t) exp(-d_perp^2/(2 sigma^2)) e_jet

    d_perp:
        distance from point to jet axis.
    """
    pts = np.asarray(points_world, dtype=float)
    u = np.zeros_like(pts)

    if not jet.time_active(t):
        return u, 0.0, 0.0

    if rng is None or jet.noise_std <= 0:
        eps = 0.0
    else:
        eps = rng.normal(0.0, jet.noise_std)

    U0 = jet.Umax * (1.0 + eps)
    U0 = max(U0, 0.0)

    center = jet.center_vector()
    e = jet.direction_unit()

    rel = pts - center.reshape(1, 3)
    axial = (rel @ e).reshape(-1, 1) * e.reshape(1, 3)
    perp = rel - axial
    d2 = np.sum(perp * perp, axis=1)

    sigma = max(jet.sigma, 1e-9)
    profile = np.exp(-d2 / (2.0 * sigma * sigma))

    u = U0 * profile.reshape(-1, 1) * e.reshape(1, 3)

    return u, U0, eps


def local_surface_velocity(v_com, omega, ri_world):
    omega_arr = np.tile(omega.reshape(1, 3), (len(ri_world), 1))
    return v_com.reshape(1, 3) + np.cross(omega_arr, ri_world)


def compute_jet_force_torque(obj, r_com, v_com, q, omega, t, jet, config, rng=None):
    p_world, ri_world = transform_surface_points(obj, r_com, q)
    v_local = local_surface_velocity(v_com, omega, ri_world)
    u_jet, U0, eps = gaussian_jet_velocity(p_world, t, jet, rng=rng)

    u_rel = u_jet - v_local
    speed_rel = np.linalg.norm(u_rel, axis=1)

    coeff = 0.5 * config.rho_air * obj.cd * obj.area_weights * speed_rel
    F_i = coeff.reshape(-1, 1) * u_rel

    F_jet = F_i.sum(axis=0)
    tau_jet = np.cross(ri_world, F_i).sum(axis=0)

    return {
        "p_world": p_world,
        "ri_world": ri_world,
        "v_local": v_local,
        "u_jet": u_jet,
        "u_rel": u_rel,
        "F_i": F_i,
        "F_jet": F_jet,
        "tau_jet": tau_jet,
        "U0": U0,
        "eps": eps
    }


def ambient_drag_force(obj, v_com, config):
    speed = np.linalg.norm(v_com)
    if speed < 1e-15:
        return np.zeros(3)

    return -0.5 * config.rho_air * obj.cd * obj.area_ref * speed * v_com


def rotational_update(obj, q, omega, tau_world, dt):
    R = quat_to_rotmat(q)
    I_world = R @ obj.inertia_body @ R.T
    Iomega = I_world @ omega
    rhs = tau_world - np.cross(omega, Iomega)

    try:
        alpha = np.linalg.solve(I_world, rhs)
    except np.linalg.LinAlgError:
        alpha = np.linalg.pinv(I_world) @ rhs

    omega_new = omega + dt * alpha
    q_new = update_quaternion(q, omega_new, dt)

    return q_new, omega_new, I_world, alpha


# 5. Main 3D simulation


In [ ]:
# ============================================================
# 5. Main 3D simulation
# ============================================================

def simulate_3d_rigid_body(
    obj,
    jet,
    geom,
    config,
    initial,
    force_jet_off=False,
    seed=None
):
    rng = np.random.default_rng(config.seed if seed is None else seed)

    dt = config.dt
    t = 0.0

    r = np.array([initial.x0, initial.y0, initial.z0], dtype=float)
    v = np.array([config.conveyor_speed, 0.0, 0.0], dtype=float)

    q = quat_normalize(initial.q0)
    omega = np.array(initial.omega0, dtype=float)

    J = np.zeros(3)
    L = np.zeros(3)

    phase = "contact"
    detached = False
    detach_reason = "none"

    rows = []

    while t <= config.t_max + 1e-12:
        jet_used = JetField(
            Umax=0.0 if force_jet_off else jet.Umax,
            sigma=jet.sigma,
            t_on=jet.t_on,
            duration=jet.duration,
            x_center=jet.x_center,
            y_center=jet.y_center,
            z_center=jet.z_center,
            azimuth_deg=jet.azimuth_deg,
            elevation_deg=jet.elevation_deg,
            noise_std=0.0 if force_jet_off else jet.noise_std
        )

        force_info = compute_jet_force_torque(
            obj=obj,
            r_com=r,
            v_com=v,
            q=q,
            omega=omega,
            t=t,
            jet=jet_used,
            config=config,
            rng=rng
        )

        F_jet = force_info["F_jet"]
        tau_jet = force_info["tau_jet"]

        F_drag = ambient_drag_force(obj, v, config) if phase == "flight" else np.zeros(3)
        F_gravity = np.array([0.0, 0.0, -obj.mass * config.g]) if phase == "flight" else np.zeros(3)

        F_total = F_jet + F_drag + F_gravity
        tau_total = tau_jet

        if jet_used.time_active(t):
            J += F_jet * dt
            L += tau_jet * dt

        euler_deg = quat_to_euler_deg(q)

        rows.append({
            "t": t,
            "phase": phase,
            "detached": detached,
            "detach_reason": detach_reason,

            "x": r[0],
            "y": r[1],
            "z": r[2],

            "vx": v[0],
            "vy": v[1],
            "vz": v[2],

            "q0": q[0],
            "q1": q[1],
            "q2": q[2],
            "q3": q[3],

            "roll_deg": euler_deg[0],
            "pitch_deg": euler_deg[1],
            "yaw_deg": euler_deg[2],

            "omega_x": omega[0],
            "omega_y": omega[1],
            "omega_z": omega[2],
            "omega_norm": np.linalg.norm(omega),

            "Fjet_x": F_jet[0],
            "Fjet_y": F_jet[1],
            "Fjet_z": F_jet[2],
            "Fjet_norm": np.linalg.norm(F_jet),

            "Fdrag_x": F_drag[0],
            "Fdrag_y": F_drag[1],
            "Fdrag_z": F_drag[2],

            "Ftotal_x": F_total[0],
            "Ftotal_y": F_total[1],
            "Ftotal_z": F_total[2],

            "tau_x": tau_jet[0],
            "tau_y": tau_jet[1],
            "tau_z": tau_jet[2],
            "tau_norm": np.linalg.norm(tau_jet),

            "Jx": J[0],
            "Jy": J[1],
            "Jz": J[2],
            "J_norm": np.linalg.norm(J),

            "Lx": L[0],
            "Ly": L[1],
            "Lz": L[2],
            "L_norm": np.linalg.norm(L),

            "U0": force_info["U0"],
            "jet_eps": force_info["eps"],
            "jet_time_active": jet_used.time_active(t)
        })

        if phase == "landed":
            break

        q_new, omega_new, I_world, alpha = rotational_update(
            obj=obj,
            q=q,
            omega=omega,
            tau_world=tau_total,
            dt=dt
        )

        if phase == "contact":
            # Conveyor-constrained phase:
            # x follows conveyor, z fixed at conveyor height.
            # y may move due to jet because y is lateral belt coordinate.
            a_y = F_jet[1] / obj.mass

            v_new = v.copy()
            r_new = r.copy()

            v_new[0] = config.conveyor_speed
            v_new[1] = v[1] + a_y * dt
            v_new[2] = 0.0

            r_new[0] = r[0] + config.conveyor_speed * dt
            r_new[1] = r[1] + v_new[1] * dt
            r_new[2] = initial.z0

            if np.linalg.norm(J) >= config.J_detach:
                phase_new = "flight"
                detached_new = True
                detach_reason_new = "jet impulse threshold"

            elif r_new[0] >= geom.x_end:
                phase_new = "flight"
                detached_new = True
                detach_reason_new = "conveyor end"

            else:
                phase_new = "contact"
                detached_new = detached
                detach_reason_new = detach_reason

        elif phase == "flight":
            a = F_total / obj.mass

            v_new = v + a * dt
            r_new = r + v_new * dt

            phase_new = "flight"
            detached_new = True
            detach_reason_new = detach_reason

            if r[2] > geom.landing_z and r_new[2] <= geom.landing_z:
                denom = r_new[2] - r[2]
                alpha_cross = 0.0 if abs(denom) < 1e-15 else (geom.landing_z - r[2]) / denom
                alpha_cross = float(np.clip(alpha_cross, 0.0, 1.0))

                r_new = r + alpha_cross * (r_new - r)
                v_new = v + alpha_cross * (v_new - v)
                r_new[2] = geom.landing_z

                phase_new = "landed"
                detached_new = True
                detach_reason_new = detach_reason

        else:
            break

        r = r_new
        v = v_new
        q = q_new
        omega = omega_new

        phase = phase_new
        detached = detached_new
        detach_reason = detach_reason_new

        t += dt

    df = pd.DataFrame(rows)

    return SimulationResult3D(
        data=df,
        obj=obj,
        jet=jet,
        geom=geom,
        config=config,
        initial=initial
    )


# 6 Visualization and 3D animation

## 6A. Static visualization functions


In [ ]:
# ============================================================
# 6. Visualization functions
# ============================================================

def plot_body_point_cloud(obj):
    pts = obj.points_body

    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection="3d")

    s = 20 + 4000 * obj.area_weights / np.max(obj.area_weights)
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=s, alpha=0.75)

    ax.set_xlabel("body x [m]")
    ax.set_ylabel("body y [m]")
    ax.set_zlabel("body z [m]")
    ax.set_title(f"Object surface point cloud: {obj.name}")

    max_range = max(np.ptp(pts, axis=0).max(), 1e-3)
    center = pts.mean(axis=0)

    ax.set_xlim(center[0] - max_range/2, center[0] + max_range/2)
    ax.set_ylim(center[1] - max_range/2, center[1] + max_range/2)
    ax.set_zlim(center[2] - max_range/2, center[2] + max_range/2)

    plt.show()

    print("Object summary")
    display(pd.DataFrame([{
        "object": obj.name,
        "mass_kg": obj.mass,
        "cd": obj.cd,
        "num_surface_points": len(obj.points_body),
        "area_ref_m2": obj.area_ref,
        "Ix_body": obj.inertia_body[0, 0],
        "Iy_body": obj.inertia_body[1, 1],
        "Iz_body": obj.inertia_body[2, 2],
    }]))


def plot_gaussian_jet_field(jet, geom, sigmas=None):
    """
    Plot jet field magnitude in the x-z plane at y = jet.y_center.
    For angled jets, this is a 2D slice through the 3D jet field.
    """
    if sigmas is None:
        sigmas = [jet.sigma]

    x_min = geom.x_start - 0.1
    x_max = max(0.8, geom.x_end + 0.8)

    x = np.linspace(x_min, x_max, 150)
    z = np.linspace(geom.landing_z, geom.height + 0.40, 120)
    X, Z = np.meshgrid(x, z)
    Y = np.ones_like(X) * jet.y_center

    points = np.column_stack([X.ravel(), Y.ravel(), Z.ravel()])

    fig, axes = plt.subplots(1, len(sigmas), figsize=(5*len(sigmas), 4), squeeze=False)

    for ax, sigma in zip(axes[0], sigmas):
        jet_tmp = JetField(
            Umax=jet.Umax,
            sigma=sigma,
            t_on=jet.t_on,
            duration=jet.duration,
            x_center=jet.x_center,
            y_center=jet.y_center,
            z_center=jet.z_center,
            azimuth_deg=jet.azimuth_deg,
            elevation_deg=jet.elevation_deg,
            noise_std=0.0
        )

        u, _, _ = gaussian_jet_velocity(points, jet.t_on + 0.5*jet.duration, jet_tmp)
        mag = np.linalg.norm(u, axis=1).reshape(X.shape)

        cs = ax.contourf(X, Z, mag, levels=30)
        ax.scatter([jet.x_center], [jet.z_center], s=70, marker="x", label="jet center")
        ax.axhline(geom.height, linestyle="--", linewidth=1, label="conveyor height")
        ax.axvline(geom.x_start, linestyle=":", linewidth=1, label="conveyor start")
        ax.axvline(geom.x_end, linestyle="-.", linewidth=1, label="conveyor end")

        e = jet.direction_unit()
        ax.arrow(
            jet.x_center,
            jet.z_center,
            0.12 * e[0],
            0.12 * e[2],
            head_width=0.015,
            length_includes_head=True
        )

        ax.set_xlabel("x [m]")
        ax.set_ylabel("z [m]")
        ax.set_title(f"Jet field slice, sigma={sigma:.3f} m")
        ax.legend(fontsize=7)
        fig.colorbar(cs, ax=ax, label="|u_jet| [m/s]")

    plt.tight_layout()
    plt.show()


def plot_result_3d(result):
    df = result.data
    obj = result.obj
    jet = result.jet
    geom = result.geom
    config = result.config

    x_lo = min(config.x_pass_min, config.x_pass_max)
    x_hi = max(config.x_pass_min, config.x_pass_max)

    class_label = "PASS-class" if config.object_should_pass else "REJECT-class"

    summary = pd.DataFrame([{
        "object": obj.name,
        "class": class_label,
        "landing_x_m": result.landing_x,
        "landing_y_m": result.landing_y,
        "landing_z_m": result.landing_z,
        "in_x_pass_region": result.in_pass_region(),
        "sorting_success": result.sorting_success(),
        "x_pass_min_m": x_lo,
        "x_pass_max_m": x_hi,
        "flight_time_s": result.flight_time,
        "detached": result.detached,
        "detachment_time_s": result.detachment_time,
        "detachment_x_m": result.detachment_x,
        "J_norm_Ns": result.J_norm,
        "L_norm_Nms": result.L_norm,
        "max_omega_rad_s": result.max_omega,
    }])
    display(summary)

    # 3D trajectory
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")

    ax.plot(df["x"], df["y"], df["z"], linewidth=2.5, label="COM trajectory")
    ax.scatter(df["x"].iloc[0], df["y"].iloc[0], df["z"].iloc[0], s=60, label="start")
    ax.scatter(result.landing_x, result.landing_y, result.landing_z, s=80, label="landing")

    # Fixed conveyor outline
    xs = [geom.x_start, geom.x_end, geom.x_end, geom.x_start, geom.x_start]
    ys = [-geom.width/2, -geom.width/2, geom.width/2, geom.width/2, -geom.width/2]
    zs = [geom.height] * 5
    ax.plot(xs, ys, zs, linestyle="--", linewidth=2, label="fixed conveyor outline")

    # Pass region on landing plane as two x boundary lines
    y_span = max(abs(df["y"]).max() + 0.15, geom.width/2 + 0.15)
    ax.plot(
        [x_lo, x_lo],
        [-y_span, y_span],
        [geom.landing_z, geom.landing_z],
        linestyle=":",
        linewidth=2,
        label="x pass min"
    )
    ax.plot(
        [x_hi, x_hi],
        [-y_span, y_span],
        [geom.landing_z, geom.landing_z],
        linestyle=":",
        linewidth=2,
        label="x pass max"
    )

    # Jet center and direction
    e = jet.direction_unit()
    ax.scatter([jet.x_center], [jet.y_center], [jet.z_center], s=80, marker="x", label="jet center")
    ax.quiver(
        jet.x_center,
        jet.y_center,
        jet.z_center,
        0.18*e[0],
        0.18*e[1],
        0.18*e[2],
        linewidth=2,
        label="jet direction"
    )

    # Object orientation snapshots
    snapshot_idx = np.linspace(0, len(df)-1, min(5, len(df))).astype(int)
    for idx in snapshot_idx:
        row = df.iloc[idx]
        r = row[["x", "y", "z"]].to_numpy(dtype=float)
        q = row[["q0", "q1", "q2", "q3"]].to_numpy(dtype=float)
        pts_world, _ = transform_surface_points(obj, r, q)
        ax.scatter(
            pts_world[:, 0],
            pts_world[:, 1],
            pts_world[:, 2],
            s=5,
            alpha=0.25
        )

    ax.set_xlabel("x: conveyor direction [m]")
    ax.set_ylabel("y: belt width / lateral coordinate [m]")
    ax.set_zlabel("z: vertical [m]")
    ax.set_title("3D COM trajectory with x-pass region")
    ax.legend(fontsize=8)
    ax.view_init(elev=22, azim=-55)
    plt.show()

    # Top view x-y
    fig, ax = plt.subplots(figsize=(8, 5))
    
    # plotting용 y 값 정리
    y_plot = df["y"].copy().to_numpy()
    y_plot[np.abs(y_plot) < 1e-10] = 0.0
    
    landing_y_plot = 0.0 if abs(result.landing_y) < 1e-10 else result.landing_y
    jet_y_plot = 0.0 if abs(jet.y_center) < 1e-10 else jet.y_center
    
    ax.plot(df["x"], y_plot, linewidth=2.5, label="x-y trajectory")
    ax.axvline(geom.x_start, linestyle=":", label="conveyor start")
    ax.axvline(geom.x_end, linestyle="-.", label="conveyor end")
    ax.axvspan(x_lo, x_hi, alpha=0.18, label="x-pass region")
    ax.scatter(result.landing_x, landing_y_plot, s=80, label="landing")
    ax.scatter(jet.x_center, jet_y_plot, s=70, marker="x", label="jet center x-y")
    
    # x축 범위
    x_min = min(geom.x_start, df["x"].min(), x_lo) - 0.05
    x_max = max(geom.x_end, df["x"].max(), x_hi) + 0.10
    ax.set_xlim(x_min, x_max)
    
    # y축 범위: 너무 작은 수치오차 때문에 1e-20 스케일이 뜨지 않도록 강제 설정
    y_abs_data = float(np.max(np.abs(y_plot))) if len(y_plot) > 0 else 0.0
    
    if y_abs_data < 1e-8:
        # 거의 y 이동이 없으면 conveyor width 기준으로 보기 좋게 고정
        y_half_range = max(0.5 * geom.width, 0.15)
    else:
        # 실제 y 이동이 있으면 그 범위를 반영하되 약간 여유 추가
        y_half_range = max(1.2 * y_abs_data + 0.02, 0.15)
    
    ax.set_ylim(-y_half_range, y_half_range)
    
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title("Top view: x-y trajectory and x-pass region")
    ax.grid(True, alpha=0.35)
    ax.legend(fontsize=8)
    plt.show()

    # x-z side view
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df["x"], df["z"], linewidth=2.5, label="x-z trajectory")
    ax.axvline(geom.x_start, linestyle=":", label="conveyor start")
    ax.axvline(geom.x_end, linestyle="-.", label="conveyor end")
    ax.axvspan(x_lo, x_hi, alpha=0.18, label="x-pass region")
    ax.axhline(geom.height, linestyle="--", label="conveyor height")
    ax.axhline(geom.landing_z, linestyle="-", label="landing plane")
    ax.scatter(result.landing_x, result.landing_z, s=80, label="landing")
    ax.scatter(jet.x_center, jet.z_center, s=70, marker="x", label="jet center x-z")

    ax.set_xlabel("x [m]")
    ax.set_ylabel("z [m]")
    ax.set_title("Side view: x-z trajectory and x-pass region")
    ax.grid(True, alpha=0.35)
    ax.legend(fontsize=8)
    plt.show()

    # Force and torque time series
    fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)

    axes[0].plot(df["t"], df["Fjet_x"], label="Fjet_x")
    axes[0].plot(df["t"], df["Fjet_y"], label="Fjet_y")
    axes[0].plot(df["t"], df["Fjet_z"], label="Fjet_z")
    axes[0].plot(df["t"], df["Fjet_norm"], label="|Fjet|", linewidth=2)
    axes[0].set_ylabel("Force [N]")
    axes[0].set_title("Jet force time series")
    axes[0].grid(True, alpha=0.35)
    axes[0].legend(fontsize=8)

    axes[1].plot(df["t"], df["tau_x"], label="tau_x")
    axes[1].plot(df["t"], df["tau_y"], label="tau_y")
    axes[1].plot(df["t"], df["tau_z"], label="tau_z")
    axes[1].plot(df["t"], df["tau_norm"], label="|tau|", linewidth=2)
    axes[1].set_ylabel("Torque [N m]")
    axes[1].set_title("Jet torque time series")
    axes[1].grid(True, alpha=0.35)
    axes[1].legend(fontsize=8)

    axes[2].plot(df["t"], df["omega_norm"], label="|omega|")
    axes[2].plot(df["t"], df["J_norm"], label="|J|")
    axes[2].plot(df["t"], df["L_norm"], label="|L|")
    axes[2].set_xlabel("time [s]")
    axes[2].set_ylabel("value")
    axes[2].set_title("Angular velocity and impulse history")
    axes[2].grid(True, alpha=0.35)
    axes[2].legend(fontsize=8)

    for ax in axes:
        ax.axvspan(jet.t_on, jet.t_on + jet.duration, alpha=0.12)

    plt.tight_layout()
    plt.show()

    # Orientation
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(df["t"], df["roll_deg"], label="roll")
    ax.plot(df["t"], df["pitch_deg"], label="pitch")
    ax.plot(df["t"], df["yaw_deg"], label="yaw")
    ax.set_xlabel("time [s]")
    ax.set_ylabel("Euler angle [deg]")
    ax.set_title("Orientation history from quaternion")
    ax.grid(True, alpha=0.35)
    ax.legend(fontsize=8)
    plt.show()
    # Angular velocity components
    fig, ax = plt.subplots(figsize=(9, 4))
    
    ax.plot(df["t"], df["omega_x"], label=r"$\omega_x$")
    ax.plot(df["t"], df["omega_y"], label=r"$\omega_y$")
    ax.plot(df["t"], df["omega_z"], label=r"$\omega_z$")
    ax.plot(df["t"], df["omega_norm"], linestyle="dotted", label=r"$|\omega|$", linewidth=2)
    
    ax.axhline(0, linewidth=1)
    ax.set_xlabel("time [s]")
    ax.set_ylabel("angular velocity [rad/s]")
    ax.set_title("Angular velocity components")
    ax.grid(True, alpha=0.35)
    ax.legend(fontsize=8)
    
    plt.show()

## 6B. 3D animation function with JET ON/OFF line color


In [ ]:
def animate_result_3d(result, every_n=8, interval=45):
    """
    3D animation with simple jet ON/OFF visual feedback.

    Visual behavior:
    - JET OFF: jet axis is gray, thin, and semi-transparent.
    - JET ON : jet axis becomes red and thicker.
    - No jet pulse marker is used.
    """
    df = result.data.copy()
    obj = result.obj
    jet = result.jet
    geom = result.geom
    config = result.config

    x_lo = min(config.x_pass_min, config.x_pass_max)
    x_hi = max(config.x_pass_min, config.x_pass_max)

    df_anim = df.iloc[::every_n, :].reset_index(drop=True)
    if df_anim["t"].iloc[-1] != df["t"].iloc[-1]:
        df_anim = pd.concat([df_anim, df.iloc[[-1]]], ignore_index=True)

    fig = plt.figure(figsize=(8.8, 7.2))
    ax = fig.add_subplot(111, projection="3d")
    # Leave extra top margin so the title, status box, and legend do not overlap.
    fig.subplots_adjust(top=0.84, left=0.07, right=0.93)

    x_min = geom.x_start - 0.10
    x_max = max(0.8, df["x"].max() + 0.2, x_hi + 0.2)
    y_abs = max(geom.width/2 + 0.2, abs(df["y"]).max() + 0.2)
    z_max = max(geom.height + 0.35, df["z"].max() + 0.2)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(-y_abs, y_abs)
    ax.set_zlim(geom.landing_z, z_max)

    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    ax.set_title("Animated 3D rigid-body trajectory with jet state", pad=24)
    ax.view_init(elev=22, azim=-55)

    ax.plot(df["x"], df["y"], df["z"], alpha=0.25, linewidth=1.5, label="full COM path")

    # Fixed conveyor
    xs = [geom.x_start, geom.x_end, geom.x_end, geom.x_start, geom.x_start]
    ys = [-geom.width/2, -geom.width/2, geom.width/2, geom.width/2, -geom.width/2]
    zs = [geom.height] * 5
    ax.plot(xs, ys, zs, linestyle="--", linewidth=1.5, label="fixed conveyor")

    # x pass boundaries
    ax.plot(
        [x_lo, x_lo],
        [-y_abs, y_abs],
        [geom.landing_z, geom.landing_z],
        linestyle=":",
        linewidth=2,
        label="x pass min"
    )
    ax.plot(
        [x_hi, x_hi],
        [-y_abs, y_abs],
        [geom.landing_z, geom.landing_z],
        linestyle=":",
        linewidth=2,
        label="x pass max"
    )

    # Jet center and dynamic jet-state visualization
    e = jet.direction_unit()
    jet_origin = np.array([jet.x_center, jet.y_center, jet.z_center], dtype=float)
    jet_len = 0.22
    jet_tip = jet_origin + jet_len * e

    ax.scatter(
        [jet_origin[0]],
        [jet_origin[1]],
        [jet_origin[2]],
        s=70,
        marker="x",
        label="jet center"
    )

    # Faint reference line: fixed jet axis
    ax.plot(
        [jet_origin[0], jet_tip[0]],
        [jet_origin[1], jet_tip[1]],
        [jet_origin[2], jet_tip[2]],
        linestyle="--",
        linewidth=1.0,
        alpha=0.35,
        label="jet axis reference"
    )

    # Dynamic jet line: color and thickness are updated every frame
    jet_line, = ax.plot(
        [jet_origin[0], jet_tip[0]],
        [jet_origin[1], jet_tip[1]],
        [jet_origin[2], jet_tip[2]],
        linewidth=2.0,
        alpha=0.35,
        label="jet state"
    )

    com_line, = ax.plot([], [], [], linewidth=2.2, label="animated COM path")
    com_point, = ax.plot([], [], [], "o", markersize=8, label="COM")
    surf = ax.scatter([], [], [], s=10, alpha=0.75)

    # Status box is placed in the top-left of the whole figure,
    # not inside the 3D axes, to avoid overlapping the title or legend.
    text = fig.text(
        0.015,
        0.985,
        "",
        ha="left",
        va="top",
        fontsize=9,
        bbox=dict(
            boxstyle="round,pad=0.35",
            facecolor="white",
            edgecolor="0.5",
            alpha=0.88
        )
    )

    # Legend is fixed at the top-right of the whole figure.
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper right",
        bbox_to_anchor=(0.985, 0.985),
        fontsize=8,
        framealpha=0.88
    )

    def set_line_3d(line, p0, p1):
        line.set_data([p0[0], p1[0]], [p0[1], p1[1]])
        line.set_3d_properties([p0[2], p1[2]])

    def init():
        com_line.set_data([], [])
        com_line.set_3d_properties([])
        com_point.set_data([], [])
        com_point.set_3d_properties([])
        surf._offsets3d = ([], [], [])

        set_line_3d(jet_line, jet_origin, jet_tip)
        jet_line.set_color("gray")
        jet_line.set_linewidth(2.0)
        jet_line.set_alpha(0.35)

        text.set_text("")
        return com_line, com_point, surf, jet_line, text

    def update(frame_idx):
        partial = df_anim.iloc[:frame_idx + 1]
        row = df_anim.iloc[frame_idx]

        com_line.set_data(partial["x"], partial["y"])
        com_line.set_3d_properties(partial["z"])

        com_point.set_data([row["x"]], [row["y"]])
        com_point.set_3d_properties([row["z"]])

        r = row[["x", "y", "z"]].to_numpy(dtype=float)
        q = row[["q0", "q1", "q2", "q3"]].to_numpy(dtype=float)
        pts_world, _ = transform_surface_points(obj, r, q)
        surf._offsets3d = (pts_world[:, 0], pts_world[:, 1], pts_world[:, 2])

        jet_active_now = bool(row["jet_time_active"])

        if jet_active_now:
            jet_line.set_color("red")
            jet_line.set_linewidth(4.5)
            jet_line.set_alpha(0.95)
            jet_state_text = "JET ON"
        else:
            jet_line.set_color("gray")
            jet_line.set_linewidth(2.0)
            jet_line.set_alpha(0.35)
            jet_state_text = "JET OFF"

        in_region_now = result.in_pass_region()
        success_now = result.sorting_success()

        text.set_text(
            f"t = {row['t']:.3f} s\n"
            f"phase = {row['phase']}\n"
            f"jet = {jet_state_text}\n"
            f"x_land = {result.landing_x:.3f} m\n"
            f"in pass region = {in_region_now}\n"
            f"success = {success_now}"
        )

        return com_line, com_point, surf, jet_line, text

    anim = FuncAnimation(
        fig,
        update,
        frames=len(df_anim),
        init_func=init,
        interval=interval,
        blit=False,
        repeat=True
    )

    html = anim.to_jshtml()
    display(HTML(html))
    plt.close(fig)
    


# 7. Analysis functions


## 7.1 run_center_vs_edge_comparison


In [ ]:
def run_center_vs_edge_comparison(base_obj, base_jet, geom, config, initial):
    """
    Compare center-hit and edge-hit by changing jet z center.
    The success criterion is still x-pass-region based.
    """
    x_hit = initial.x0 + config.conveyor_speed * base_jet.t_on

    center_jet = JetField(
        Umax=base_jet.Umax,
        sigma=base_jet.sigma,
        t_on=base_jet.t_on,
        duration=base_jet.duration,
        x_center=x_hit,
        y_center=initial.y0,
        z_center=initial.z0,
        azimuth_deg=base_jet.azimuth_deg,
        elevation_deg=base_jet.elevation_deg,
        noise_std=0.0
    )

    edge_jet = JetField(
        Umax=base_jet.Umax,
        sigma=base_jet.sigma,
        t_on=base_jet.t_on,
        duration=base_jet.duration,
        x_center=x_hit,
        y_center=initial.y0,
        z_center=initial.z0 + 0.055,
        azimuth_deg=base_jet.azimuth_deg,
        elevation_deg=base_jet.elevation_deg,
        noise_std=0.0
    )

    res_center = simulate_3d_rigid_body(base_obj, center_jet, geom, config, initial, seed=config.seed)
    res_edge = simulate_3d_rigid_body(base_obj, edge_jet, geom, config, initial, seed=config.seed)

    comp = pd.DataFrame([
        {
            "case": "center hit",
            "x_land_m": res_center.landing_x,
            "in_pass_region": res_center.in_pass_region(),
            "success": res_center.sorting_success(),
            "J_norm_Ns": res_center.J_norm,
            "L_norm_Nms": res_center.L_norm,
            "max_omega_rad_s": res_center.max_omega
        },
        {
            "case": "edge hit",
            "x_land_m": res_edge.landing_x,
            "in_pass_region": res_edge.in_pass_region(),
            "success": res_edge.sorting_success(),
            "J_norm_Ns": res_edge.J_norm,
            "L_norm_Nms": res_edge.L_norm,
            "max_omega_rad_s": res_edge.max_omega
        }
    ])

    display(comp)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    axes[0].bar(comp["case"], comp["x_land_m"])
    axes[0].axhspan(min(config.x_pass_min, config.x_pass_max), max(config.x_pass_min, config.x_pass_max), alpha=0.18)
    axes[0].set_title("Landing x")
    axes[0].set_ylabel("x_land [m]")
    axes[0].grid(True, axis="y", alpha=0.35)

    axes[1].bar(comp["case"], comp["L_norm_Nms"])
    axes[1].set_title("Angular impulse |L|")
    axes[1].set_ylabel("N m s")
    axes[1].grid(True, axis="y", alpha=0.35)

    axes[2].bar(comp["case"], comp["max_omega_rad_s"])
    axes[2].set_title("Max angular velocity")
    axes[2].set_ylabel("rad/s")
    axes[2].grid(True, axis="y", alpha=0.35)

    plt.tight_layout()
    plt.show()

    return comp, res_center, res_edge


## 7.2 run_hit_offset_sweep


In [ ]:
def run_hit_offset_sweep(base_obj, base_jet, geom, config, initial):
    offsets = np.linspace(-0.08, 0.08, 17)
    rows = []

    x_hit = initial.x0 + config.conveyor_speed * base_jet.t_on

    for offset in offsets:
        jet = JetField(
            Umax=base_jet.Umax,
            sigma=base_jet.sigma,
            t_on=base_jet.t_on,
            duration=base_jet.duration,
            x_center=x_hit,
            y_center=base_jet.y_center,
            z_center=initial.z0 + offset,
            azimuth_deg=base_jet.azimuth_deg,
            elevation_deg=base_jet.elevation_deg,
            noise_std=0.0
        )

        res = simulate_3d_rigid_body(base_obj, jet, geom, config, initial, seed=config.seed)

        rows.append({
            "offset_z_m": offset,
            "x_land_m": res.landing_x,
            "in_pass_region": res.in_pass_region(),
            "success": res.sorting_success(),
            "J_norm_Ns": res.J_norm,
            "L_norm_Nms": res.L_norm,
            "max_omega_rad_s": res.max_omega,
        })

    df = pd.DataFrame(rows)
    display(df)

    x_lo = min(config.x_pass_min, config.x_pass_max)
    x_hi = max(config.x_pass_min, config.x_pass_max)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0, 0].plot(df["offset_z_m"], df["x_land_m"], marker="o")
    axes[0, 0].axhspan(x_lo, x_hi, alpha=0.18)
    axes[0, 0].set_xlabel("hit offset in z [m]")
    axes[0, 0].set_ylabel("x_land [m]")
    axes[0, 0].set_title("Landing x vs hit offset")

    axes[0, 1].plot(df["offset_z_m"], df["max_omega_rad_s"], marker="o")
    axes[0, 1].set_xlabel("hit offset in z [m]")
    axes[0, 1].set_ylabel("|omega|max [rad/s]")
    axes[0, 1].set_title("Max angular velocity vs hit offset")

    axes[1, 0].plot(df["offset_z_m"], df["J_norm_Ns"], marker="o")
    axes[1, 0].set_xlabel("hit offset in z [m]")
    axes[1, 0].set_ylabel("|J| [N s]")
    axes[1, 0].set_title("Linear impulse vs hit offset")

    axes[1, 1].plot(df["offset_z_m"], df["L_norm_Nms"], marker="o")
    axes[1, 1].set_xlabel("hit offset in z [m]")
    axes[1, 1].set_ylabel("|L| [N m s]")
    axes[1, 1].set_title("Angular impulse vs hit offset")

    for ax in axes.ravel():
        ax.grid(True, alpha=0.35)

    plt.tight_layout()
    plt.show()

    return df


## 7.3 run_timing_duration_sweep


In [ ]:
def run_timing_duration_sweep(base_obj, base_jet, geom, config, initial):
    ton_values = np.linspace(max(0.0, base_jet.t_on - 0.18), base_jet.t_on + 0.25, 12)
    duration_values = np.linspace(0.05, 0.35, 10)

    records = []

    for ton in ton_values:
        for dur in duration_values:
            jet = JetField(
                Umax=base_jet.Umax,
                sigma=base_jet.sigma,
                t_on=float(ton),
                duration=float(dur),
                x_center=base_jet.x_center,
                y_center=base_jet.y_center,
                z_center=base_jet.z_center,
                azimuth_deg=base_jet.azimuth_deg,
                elevation_deg=base_jet.elevation_deg,
                noise_std=0.0
            )

            res = simulate_3d_rigid_body(base_obj, jet, geom, config, initial, seed=config.seed)

            records.append({
                "t_on_s": ton,
                "duration_s": dur,
                "x_land_m": res.landing_x,
                "in_pass_region": res.in_pass_region(),
                "success": res.sorting_success(),
                "J_norm_Ns": res.J_norm,
                "L_norm_Nms": res.L_norm,
                "max_omega_rad_s": res.max_omega,
            })

    df = pd.DataFrame(records)
    display(df.head())

    x_grid = df.pivot(index="duration_s", columns="t_on_s", values="x_land_m")
    success_grid = df.pivot(index="duration_s", columns="t_on_s", values="success").astype(float)
    J_grid = df.pivot(index="duration_s", columns="t_on_s", values="J_norm_Ns")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    for ax, grid, title, label in [
        (axes[0], x_grid, "Timing map: x_land", "x_land [m]"),
        (axes[1], success_grid, "Timing map: success", "success [-]"),
        (axes[2], J_grid, "Linear impulse map: |J|", "|J| [N s]"),
    ]:
        im = ax.imshow(
            grid.values,
            origin="lower",
            aspect="auto",
            extent=[
                grid.columns.min(),
                grid.columns.max(),
                grid.index.min(),
                grid.index.max()
            ]
        )
        ax.set_xlabel("t_on [s]")
        ax.set_ylabel("duration [s]")
        ax.set_title(title)
        fig.colorbar(im, ax=ax, label=label)

    plt.tight_layout()
    plt.show()

    return df


## 7.4 run_noise_trials


In [ ]:
def run_noise_trials(base_obj, base_jet, geom, config, initial, n_trials=50):
    cases = [
        ("deterministic", 0.0),
        ("low noise", 0.03),
        ("high noise", 0.12),
    ]

    rows = []

    for label, noise_std in cases:
        for k in range(n_trials):
            jet = JetField(
                Umax=base_jet.Umax,
                sigma=base_jet.sigma,
                t_on=base_jet.t_on,
                duration=base_jet.duration,
                x_center=base_jet.x_center,
                y_center=base_jet.y_center,
                z_center=base_jet.z_center,
                azimuth_deg=base_jet.azimuth_deg,
                elevation_deg=base_jet.elevation_deg,
                noise_std=noise_std
            )

            res = simulate_3d_rigid_body(
                base_obj,
                jet,
                geom,
                config,
                initial,
                seed=config.seed + k
            )

            rows.append({
                "case": label,
                "trial": k,
                "noise_std": noise_std,
                "x_land_m": res.landing_x,
                "y_land_m": res.landing_y,
                "in_pass_region": res.in_pass_region(),
                "success": res.sorting_success(),
                "J_norm_Ns": res.J_norm,
                "L_norm_Nms": res.L_norm,
                "max_omega_rad_s": res.max_omega
            })

    df = pd.DataFrame(rows)

    display(df.groupby("case").agg(
        x_mean=("x_land_m", "mean"),
        x_std=("x_land_m", "std"),
        y_mean=("y_land_m", "mean"),
        y_std=("y_land_m", "std"),
        success_rate=("success", "mean"),
        omega_mean=("max_omega_rad_s", "mean")
    ).reset_index())

    x_lo = min(config.x_pass_min, config.x_pass_max)
    x_hi = max(config.x_pass_min, config.x_pass_max)

    fig, ax = plt.subplots(figsize=(9, 5))

    for case in df["case"].unique():
        sub = df[df["case"] == case]
        ax.hist(sub["x_land_m"], bins=18, alpha=0.5, label=case)

    ax.axvspan(x_lo, x_hi, alpha=0.18, label="x-pass region")
    ax.set_xlabel("x_land [m]")
    ax.set_ylabel("count")
    ax.set_title("Noise sensitivity: landing x distribution")
    ax.grid(True, alpha=0.35)
    ax.legend()
    plt.show()

    return df


## 7.5 run_sorting_analysis


In [ ]:
def run_sorting_analysis(
    geom,
    base_jet,
    config,
    base_initial,
    current_shape_params,
    use_random_y=True,
    y0_min=-0.20,
    y0_max=0.20,
    n_per_type=25
):
    """
    Revised sorting analysis.

    pass-class object:
        success if x_land is inside x-pass region

    reject-class object:
        success if x_land is outside x-pass region

    Here, object classes are illustrative:
        pass-class:
            thin plate, sphere, irregular flake
        reject-class:
            rod-like object, cylinder

    y0 can be randomized because incoming objects may enter at random lateral positions.
    """
    rng = np.random.default_rng(config.seed + 999)

    object_specs = [
        ("thin plate", True),
        ("sphere", True),
        ("irregular flake", True),
        ("rod-like object", False),
        ("cylinder", False),
    ]

    rows = []

    for kind, should_pass in object_specs:
        for k in range(n_per_type):
            mass = float(rng.uniform(0.0045, 0.0100))
            cd = float(rng.uniform(0.9, 1.8))

            obj = create_rigid_object(
                kind=kind,
                mass=mass,
                cd=cd,
                length_x=current_shape_params["length_x"] * rng.uniform(0.8, 1.2),
                width_y=current_shape_params["width_y"] * rng.uniform(0.8, 1.2),
                thickness_z=current_shape_params["thickness_z"] * rng.uniform(0.8, 1.2),
                cylinder_radius=current_shape_params["cylinder_radius"] * rng.uniform(0.8, 1.2),
                cylinder_height=current_shape_params["cylinder_height"] * rng.uniform(0.8, 1.2),
                sphere_radius=current_shape_params["sphere_radius"] * rng.uniform(0.8, 1.2),
                resolution=current_shape_params["resolution"],
                irregular_seed=4 + k
            )

            jet = JetField(
                Umax=float(base_jet.Umax * rng.uniform(0.92, 1.08)),
                sigma=float(base_jet.sigma * rng.uniform(0.85, 1.15)),
                t_on=float(max(0.0, base_jet.t_on + rng.normal(0, 0.025))),
                duration=float(max(0.04, base_jet.duration + rng.normal(0, 0.015))),
                x_center=base_jet.x_center,
                y_center=base_jet.y_center,
                z_center=base_jet.z_center,
                azimuth_deg=base_jet.azimuth_deg,
                elevation_deg=base_jet.elevation_deg,
                noise_std=0.05
            )

            q0 = quat_from_euler_deg(
                roll_deg=rng.normal(0, 4),
                pitch_deg=rng.normal(0, 4),
                yaw_deg=rng.normal(0, 15)
            )

            if use_random_y:
                y0 = float(rng.uniform(y0_min, y0_max))
            else:
                y0 = base_initial.y0

            initial = InitialCondition3D(
                x0=base_initial.x0,
                y0=y0,
                z0=base_initial.z0,
                q0=q0,
                omega0=np.array([
                    rng.normal(0, 1.0),
                    rng.normal(0, 1.0),
                    rng.normal(0, 1.0),
                ])
            )

            trial_config = SimulationConfig3D(
                rho_air=config.rho_air,
                g=config.g,
                dt=config.dt,
                t_max=config.t_max,
                conveyor_speed=config.conveyor_speed,
                J_detach=config.J_detach,
                x_pass_min=config.x_pass_min,
                x_pass_max=config.x_pass_max,
                object_should_pass=should_pass,
                seed=config.seed + k
            )

            res = simulate_3d_rigid_body(
                obj,
                jet,
                geom,
                trial_config,
                initial,
                seed=trial_config.seed
            )

            in_region = res.in_pass_region()
            success = res.sorting_success()

            # Error definitions:
            # false negative: pass-class object fails to enter pass region
            # false positive: reject-class object enters pass region
            false_negative = should_pass and (not in_region)
            false_positive = (not should_pass) and in_region

            rows.append({
                "object": kind,
                "should_pass": should_pass,
                "trial": k,
                "mass_kg": mass,
                "cd": cd,
                "y0_m": y0,
                "x_land_m": res.landing_x,
                "y_land_m": res.landing_y,
                "in_pass_region": in_region,
                "success": success,
                "false_positive": false_positive,
                "false_negative": false_negative,
                "max_omega_rad_s": res.max_omega
            })

    df = pd.DataFrame(rows)

    summary = df.groupby(["object", "should_pass"]).agg(
        n=("trial", "count"),
        success_rate=("success", "mean"),
        false_positive_rate=("false_positive", "mean"),
        false_negative_rate=("false_negative", "mean"),
        mean_x_land=("x_land_m", "mean"),
        std_x_land=("x_land_m", "std"),
        mean_y_land=("y_land_m", "mean"),
        std_y_land=("y_land_m", "std"),
        mean_max_omega=("max_omega_rad_s", "mean")
    ).reset_index()

    display(summary)

    x_lo = min(config.x_pass_min, config.x_pass_max)
    x_hi = max(config.x_pass_min, config.x_pass_max)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].bar(summary["object"], summary["success_rate"])
    axes[0].set_ylim(0, 1.05)
    axes[0].set_ylabel("success rate")
    axes[0].set_title("Sorting success rate by object type")
    axes[0].grid(True, axis="y", alpha=0.35)
    axes[0].tick_params(axis="x", rotation=30)

    for obj_name in df["object"].unique():
        sub = df[df["object"] == obj_name]
        axes[1].hist(sub["x_land_m"], bins=15, alpha=0.45, label=obj_name)

    axes[1].axvspan(x_lo, x_hi, alpha=0.18, label="x-pass region")
    axes[1].set_xlabel("x_land [m]")
    axes[1].set_ylabel("count")
    axes[1].set_title("Landing x distribution by object type")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.35)

    plt.tight_layout()
    plt.show()

    return df, summary


## 7.6 run_sanity_checks


In [ ]:
def run_sanity_checks(base_obj, base_jet, geom, config, initial):
    checks = []

    # 1. Jet off reproducibility and gravity-only trajectory exists
    res_off = simulate_3d_rigid_body(
        base_obj,
        base_jet,
        geom,
        config,
        initial,
        force_jet_off=True,
        seed=config.seed
    )

    checks.append({
        "check": "gravity-only case reaches landing plane",
        "value": res_off.landing_z,
        "pass": abs(res_off.landing_z - geom.landing_z) < 1e-8,
        "criterion": "landing z == landing plane"
    })

    # 2. Zero noise repeated simulations identical
    zero_noise_jet = JetField(
        Umax=base_jet.Umax,
        sigma=base_jet.sigma,
        t_on=base_jet.t_on,
        duration=base_jet.duration,
        x_center=base_jet.x_center,
        y_center=base_jet.y_center,
        z_center=base_jet.z_center,
        azimuth_deg=base_jet.azimuth_deg,
        elevation_deg=base_jet.elevation_deg,
        noise_std=0.0
    )

    res_a = simulate_3d_rigid_body(base_obj, zero_noise_jet, geom, config, initial, seed=1)
    res_b = simulate_3d_rigid_body(base_obj, zero_noise_jet, geom, config, initial, seed=999)

    checks.append({
        "check": "zero noise -> identical repeated simulations",
        "value": abs(res_a.landing_x - res_b.landing_x),
        "pass": abs(res_a.landing_x - res_b.landing_x) < 1e-12,
        "criterion": "|x_a - x_b| < 1e-12"
    })

    # 3. Nonzero noise produces distribution
    noisy_jet = JetField(
        Umax=base_jet.Umax,
        sigma=base_jet.sigma,
        t_on=base_jet.t_on,
        duration=base_jet.duration,
        x_center=base_jet.x_center,
        y_center=base_jet.y_center,
        z_center=base_jet.z_center,
        azimuth_deg=base_jet.azimuth_deg,
        elevation_deg=base_jet.elevation_deg,
        noise_std=0.12
    )

    x_vals = []
    for k in range(20):
        res = simulate_3d_rigid_body(base_obj, noisy_jet, geom, config, initial, seed=100+k)
        x_vals.append(res.landing_x)

    x_std = float(np.std(x_vals))

    checks.append({
        "check": "nonzero noise -> landing x distribution",
        "value": x_std,
        "pass": x_std > 1e-8,
        "criterion": "std(x_land) > 1e-8"
    })

    # 4. Larger sigma affects more points
    pts_world, _ = transform_surface_points(
        base_obj,
        np.array([base_jet.x_center, base_jet.y_center, initial.z0]),
        initial.q0
    )

    jet_small = JetField(
        Umax=base_jet.Umax,
        sigma=0.03,
        t_on=base_jet.t_on,
        duration=base_jet.duration,
        x_center=base_jet.x_center,
        y_center=base_jet.y_center,
        z_center=base_jet.z_center,
        azimuth_deg=base_jet.azimuth_deg,
        elevation_deg=base_jet.elevation_deg,
        noise_std=0.0
    )

    jet_large = JetField(
        Umax=base_jet.Umax,
        sigma=0.12,
        t_on=base_jet.t_on,
        duration=base_jet.duration,
        x_center=base_jet.x_center,
        y_center=base_jet.y_center,
        z_center=base_jet.z_center,
        azimuth_deg=base_jet.azimuth_deg,
        elevation_deg=base_jet.elevation_deg,
        noise_std=0.0
    )

    u_small, _, _ = gaussian_jet_velocity(pts_world, base_jet.t_on + 0.01, jet_small)
    u_large, _, _ = gaussian_jet_velocity(pts_world, base_jet.t_on + 0.01, jet_large)

    n_small = int((np.linalg.norm(u_small, axis=1) > 0.2 * base_jet.Umax).sum())
    n_large = int((np.linalg.norm(u_large, axis=1) > 0.2 * base_jet.Umax).sum())

    checks.append({
        "check": "larger sigma -> broader jet distribution",
        "value": n_large - n_small,
        "pass": n_large >= n_small,
        "criterion": "number of affected points increases"
    })

    df = pd.DataFrame(checks)
    display(df)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(df["check"], df["pass"].astype(int))
    ax.set_xlim(0, 1.1)
    ax.set_xlabel("pass = 1, fail = 0")
    ax.set_title("Sanity check results")
    ax.grid(True, axis="x", alpha=0.35)
    plt.show()

    return df


# 8. Widgets


In [ ]:
# ============================================================
# 8. Widgets
# ============================================================

style = {"description_width": "175px"}
layout = widgets.Layout(width="460px")

# -----------------------------
# Object
# -----------------------------
object_kind_w = widgets.Dropdown(
    options=["thin plate", "rod-like object", "cylinder", "sphere", "irregular flake"],
    value="thin plate",
    description="object type",
    style=style,
    layout=layout
)

object_class_w = widgets.Dropdown(
    options=[("pass-class object", True), ("reject-class object", False)],
    value=True,
    description="object class",
    style=style,
    layout=layout
)

mass_w = widgets.FloatSlider(
    value=0.020,
    min=0.003,
    max=1.000,
    step=0.001,
    description="mass [kg]",
    style=style,
    layout=layout,
    readout_format=".4f",
    continuous_update=False
)

cd_w = widgets.FloatSlider(
    value=1.20,
    min=0.30,
    max=3.00,
    step=0.05,
    description="CD [-]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

surface_resolution_w = widgets.IntSlider(
    value=8,
    min=4,
    max=24,
    step=1,
    description="surface resolution",
    style=style,
    layout=layout,
    continuous_update=False
)

length_x_w = widgets.FloatSlider(
    value=0.100,
    min=0.020,
    max=0.250,
    step=0.005,
    description="length x [m]",
    style=style,
    layout=layout,
    readout_format=".3f",
    continuous_update=False
)

width_y_w = widgets.FloatSlider(
    value=0.065,
    min=0.010,
    max=0.180,
    step=0.005,
    description="width y [m]",
    style=style,
    layout=layout,
    readout_format=".3f",
    continuous_update=False
)

thickness_z_w = widgets.FloatSlider(
    value=0.008,
    min=0.002,
    max=0.080,
    step=0.002,
    description="thickness z [m]",
    style=style,
    layout=layout,
    readout_format=".3f",
    continuous_update=False
)

cyl_radius_w = widgets.FloatSlider(
    value=0.015,
    min=0.005,
    max=0.080,
    step=0.002,
    description="cylinder radius [m]",
    style=style,
    layout=layout,
    readout_format=".3f",
    continuous_update=False
)

cyl_height_w = widgets.FloatSlider(
    value=0.120,
    min=0.020,
    max=0.300,
    step=0.005,
    description="cylinder height [m]",
    style=style,
    layout=layout,
    readout_format=".3f",
    continuous_update=False
)

sphere_radius_w = widgets.FloatSlider(
    value=0.035,
    min=0.005,
    max=0.100,
    step=0.002,
    description="sphere radius [m]",
    style=style,
    layout=layout,
    readout_format=".3f",
    continuous_update=False
)

roll_w = widgets.FloatSlider(
    value=0.0,
    min=-60,
    max=60,
    step=1,
    description="initial roll [deg]",
    style=style,
    layout=layout,
    readout_format=".0f",
    continuous_update=False
)

pitch_w = widgets.FloatSlider(
    value=0.0,
    min=-60,
    max=60,
    step=1,
    description="initial pitch [deg]",
    style=style,
    layout=layout,
    readout_format=".0f",
    continuous_update=False
)

yaw_w = widgets.FloatSlider(
    value=0.0,
    min=-180,
    max=180,
    step=2,
    description="initial yaw [deg]",
    style=style,
    layout=layout,
    readout_format=".0f",
    continuous_update=False
)

omega_x_w = widgets.FloatSlider(
    value=0.0,
    min=-20,
    max=20,
    step=0.5,
    description="omega x [rad/s]",
    style=style,
    layout=layout,
    readout_format=".1f",
    continuous_update=False
)

omega_y_w = widgets.FloatSlider(
    value=0.0,
    min=-20,
    max=20,
    step=0.5,
    description="omega y [rad/s]",
    style=style,
    layout=layout,
    readout_format=".1f",
    continuous_update=False
)

omega_z_w = widgets.FloatSlider(
    value=0.0,
    min=-20,
    max=20,
    step=0.5,
    description="omega z [rad/s]",
    style=style,
    layout=layout,
    readout_format=".1f",
    continuous_update=False
)

# -----------------------------
# Conveyor
# -----------------------------
conveyor_width_w = widgets.FloatSlider(
    value=0.60,
    min=0.20,
    max=1.20,
    step=0.05,
    description="conveyor width [m]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

conveyor_height_w = widgets.FloatSlider(
    value=0.50,
    min=0.20,
    max=1.20,
    step=0.02,
    description="conveyor height [m]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

conveyor_speed_w = widgets.FloatSlider(
    value=1.20,
    min=0.20,
    max=3.00,
    step=0.05,
    description="conveyor speed [m/s]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

x0_w = widgets.FloatSlider(
    value=-0.20,
    min=-0.20,
    max=0.00,
    step=0.005,
    description="x0 [m]",
    style=style,
    layout=layout,
    readout_format=".3f",
    continuous_update=False
)

y0_w = widgets.FloatSlider(
    value=0.0,
    min=-0.30,
    max=0.30,
    step=0.01,
    description="y0 [m]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

# -----------------------------
# Jet
# -----------------------------
Umax_w = widgets.FloatSlider(
    value=20.0,
    min=0.0,
    max=40.0,
    step=0.5,
    description="Umax [m/s]",
    style=style,
    layout=layout,
    readout_format=".1f",
    continuous_update=False
)

sigma_w = widgets.FloatSlider(
    value=0.055,
    min=0.015,
    max=0.250,
    step=0.005,
    description="sigma [m]",
    style=style,
    layout=layout,
    readout_format=".3f",
    continuous_update=False
)

ton_w = widgets.FloatSlider(
    value=0.30,
    min=0.00,
    max=1.20,
    step=0.01,
    description="t_on [s]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

duration_w = widgets.FloatSlider(
    value=0.25,
    min=0.02,
    max=0.70,
    step=0.01,
    description="duration [s]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

jet_x_w = widgets.FloatSlider(
    value=0.10,
    min=-0.30,
    max=0.70,
    step=0.01,
    description="jet x center [m]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

jet_y_w = widgets.FloatSlider(
    value=0.00,
    min=-0.50,
    max=0.80,
    step=0.01,
    description="jet y center [m]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

jet_z_w = widgets.FloatSlider(
    value=0.30,
    min=0.05,
    max=1.20,
    step=0.01,
    description="jet z center [m]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

jet_azimuth_w = widgets.FloatSlider(
    value=0.0,
    min=-180.0,
    max=180.0,
    step=1.0,
    description="jet azimuth [deg]",
    style=style,
    layout=layout,
    readout_format=".0f",
    continuous_update=False
)

jet_elevation_w = widgets.FloatSlider(
    value=60.0,
    min=-180.0,
    max=180.0,
    step=1.0,
    description="jet elevation [deg]",
    style=style,
    layout=layout,
    readout_format=".0f",
    continuous_update=False
)

noise_w = widgets.FloatSlider(
    value=0.00,
    min=0.00,
    max=0.30,
    step=0.01,
    description="jet noise std [-]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

# -----------------------------
# Sorting and numerical
# -----------------------------
Jdetach_w = widgets.FloatSlider(
    value=0.003,
    min=0.0005,
    max=0.020,
    step=0.0005,
    description="J_detach [N s]",
    style=style,
    layout=layout,
    readout_format=".4f",
    continuous_update=False
)

xpass_min_w = widgets.FloatSlider(
    value=1.00,
    min=-5.00,
    max=5.00,
    step=0.1,
    description="x pass min [m]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

xpass_max_w = widgets.FloatSlider(
    value=1.5,
    min=-5.00,
    max=5.00,
    step=0.1,
    description="x pass max [m]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

tmax_w = widgets.FloatSlider(
    value=4.00,
    min=0.50,
    max=8.00,
    step=0.10,
    description="t_max [s]",
    style=style,
    layout=layout,
    readout_format=".1f",
    continuous_update=False
)

dt_w = widgets.FloatSlider(
    value=0.002,
    min=0.001,
    max=0.010,
    step=0.001,
    description="dt [s]",
    style=style,
    layout=layout,
    readout_format=".3f",
    continuous_update=False
)

seed_w = widgets.IntSlider(
    value=42,
    min=0,
    max=999,
    step=1,
    description="random seed",
    style=style,
    layout=layout,
    continuous_update=False
)

sort_random_y_w = widgets.Checkbox(
    value=True,
    description="sorting analysis: random y0",
    indent=False,
    layout=layout
)

sort_ymin_w = widgets.FloatSlider(
    value=-0.20,
    min=-0.60,
    max=0.60,
    step=0.01,
    description="sorting y0 min [m]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

sort_ymax_w = widgets.FloatSlider(
    value=0.20,
    min=-0.60,
    max=0.60,
    step=0.01,
    description="sorting y0 max [m]",
    style=style,
    layout=layout,
    readout_format=".2f",
    continuous_update=False
)

output = widgets.Output()


# 9. Setup builder


In [ ]:
# ============================================================
# 9. Setup builder
# ============================================================

def get_shape_params_dict():
    return {
        "length_x": length_x_w.value,
        "width_y": width_y_w.value,
        "thickness_z": thickness_z_w.value,
        "cylinder_radius": cyl_radius_w.value,
        "cylinder_height": cyl_height_w.value,
        "sphere_radius": sphere_radius_w.value,
        "resolution": surface_resolution_w.value
    }


def build_current_setup():
    shape_params = get_shape_params_dict()

    obj = create_rigid_object(
        kind=object_kind_w.value,
        mass=mass_w.value,
        cd=cd_w.value,
        length_x=length_x_w.value,
        width_y=width_y_w.value,
        thickness_z=thickness_z_w.value,
        cylinder_radius=cyl_radius_w.value,
        cylinder_height=cyl_height_w.value,
        sphere_radius=sphere_radius_w.value,
        resolution=surface_resolution_w.value,
        irregular_seed=4
    )

    geom = ConveyorGeometry(
        x_start=-0.20,
        x_end=0.00,
        width=conveyor_width_w.value,
        height=conveyor_height_w.value,
        landing_z=0.0
    )

    jet = JetField(
        Umax=Umax_w.value,
        sigma=sigma_w.value,
        t_on=ton_w.value,
        duration=duration_w.value,
        x_center=jet_x_w.value,
        y_center=jet_y_w.value,
        z_center=jet_z_w.value,
        azimuth_deg=jet_azimuth_w.value,
        elevation_deg=jet_elevation_w.value,
        noise_std=noise_w.value
    )

    config = SimulationConfig3D(
        rho_air=1.225,
        g=9.81,
        dt=dt_w.value,
        t_max=tmax_w.value,
        conveyor_speed=conveyor_speed_w.value,
        J_detach=Jdetach_w.value,
        x_pass_min=xpass_min_w.value,
        x_pass_max=xpass_max_w.value,
        object_should_pass=bool(object_class_w.value),
        seed=seed_w.value
    )

    q0 = quat_from_euler_deg(
        roll_deg=roll_w.value,
        pitch_deg=pitch_w.value,
        yaw_deg=yaw_w.value
    )

    omega0 = np.array([
        omega_x_w.value,
        omega_y_w.value,
        omega_z_w.value
    ], dtype=float)

    initial = InitialCondition3D(
        x0=x0_w.value,
        y0=y0_w.value,
        z0=conveyor_height_w.value,
        q0=q0,
        omega0=omega0
    )

    return obj, jet, geom, config, initial, shape_params


def run_current_simulation_3d():
    obj, jet, geom, config, initial, shape_params = build_current_setup()
    result = simulate_3d_rigid_body(
        obj=obj,
        jet=jet,
        geom=geom,
        config=config,
        initial=initial,
        seed=config.seed
    )
    return result


# 10. Buttons and callbacks


In [ ]:
# ============================================================
# 10. Buttons and callbacks
# ============================================================

update_btn = widgets.Button(description="Update plot", button_style="primary", icon="refresh")
animate_btn = widgets.Button(description="Animate 3D", button_style="info", icon="play")
object_cloud_btn = widgets.Button(description="Object points", button_style="", icon="cube")
jet_field_btn = widgets.Button(description="Jet field", button_style="", icon="arrows")
center_edge_btn = widgets.Button(description="Center vs edge", button_style="", icon="exchange")
offset_sweep_btn = widgets.Button(description="Hit-offset sweep", button_style="", icon="line-chart")
timing_sweep_btn = widgets.Button(description="Timing map", button_style="", icon="clock-o")
noise_btn = widgets.Button(description="Noise trials", button_style="", icon="random")
sorting_btn = widgets.Button(description="Sorting analysis", button_style="", icon="check")
sanity_btn = widgets.Button(description="Sanity checks", button_style="warning", icon="warning")
reset_btn = widgets.Button(description="Reset baseline", button_style="", icon="undo")


def set_baseline():
    object_kind_w.value = "thin plate"
    object_class_w.value = True

    mass_w.value = 0.020
    cd_w.value = 1.20
    surface_resolution_w.value = 8

    length_x_w.value = 0.100
    width_y_w.value = 0.065
    thickness_z_w.value = 0.008
    cyl_radius_w.value = 0.015
    cyl_height_w.value = 0.120
    sphere_radius_w.value = 0.035

    roll_w.value = 0.0
    pitch_w.value = 0.0
    yaw_w.value = 0.0

    omega_x_w.value = 0.0
    omega_y_w.value = 0.0
    omega_z_w.value = 0.0

    conveyor_width_w.value = 0.60
    conveyor_height_w.value = 0.50
    conveyor_speed_w.value = 1.20
    x0_w.value = -0.20
    y0_w.value = 0.0

    Umax_w.value = 20.0
    sigma_w.value = 0.055
    ton_w.value = 0.30
    duration_w.value = 0.25
    jet_x_w.value = 0.10
    jet_y_w.value = 0.00
    jet_z_w.value = 0.30
    jet_azimuth_w.value = 0.0
    jet_elevation_w.value = 60.0
    noise_w.value = 0.00

    Jdetach_w.value = 0.003
    xpass_min_w.value = 1.00
    xpass_max_w.value = 1.50
    tmax_w.value = 4.0
    dt_w.value = 0.002
    seed_w.value = 42

    sort_random_y_w.value = True
    sort_ymin_w.value = -0.20
    sort_ymax_w.value = 0.20


def update_display_static():
    with output:
        clear_output(wait=True)
        plt.close("all")

        result = run_current_simulation_3d()
        plot_result_3d(result)

        print("해석 요약")
        print("- 성공/실패 기준은 y가 아니라 landing plane에서의 x_land입니다.")
        print("- pass-class object: x_land가 x-pass region 안이면 성공입니다.")
        print("- reject-class object: x_land가 x-pass region 밖이면 성공입니다.")
        print("- y축은 넓은 컨베이어에서 object의 lateral input 위치와 jet targeting 위치를 나타냅니다.")
        print("- 컨베이어는 x = -0.2 m에서 시작해서 x = 0.0 m에서 끝나도록 고정되어 있습니다.")
        print("- jet center는 x, y, z 모두 조절 가능하며, jet 방향은 azimuth/elevation으로 조절됩니다.")
        print("- azimuth = 0 deg이면 기본 +y 방향이고, 180 또는 -180 deg이면 -y 방향입니다.")
        print("- elevation > 0이면 +z 방향 성분, elevation < 0이면 -z 방향 성분을 가집니다.")
        print("- |J_jet| > J_detach이면 impulse-based detachment가 발생합니다.")


def update_display_animation():
    with output:
        clear_output(wait=True)
        

        result = run_current_simulation_3d()

        class_label = "PASS-class" if result.config.object_should_pass else "REJECT-class"

        summary = pd.DataFrame([{
            "object": result.obj.name,
            "class": class_label,
            "landing_x_m": result.landing_x,
            "landing_y_m": result.landing_y,
            "in_x_pass_region": result.in_pass_region(),
            "sorting_success": result.sorting_success(),
            "J_norm_Ns": result.J_norm,
            "L_norm_Nms": result.L_norm,
            "max_omega_rad_s": result.max_omega,
        }])
        display(summary)

        animate_result_3d(result, every_n=8, interval=45)


def on_update_clicked(b):
    update_display_static()


def on_animate_clicked(b):
    update_display_animation()


def on_object_cloud_clicked(b):
    with output:
        clear_output(wait=True)
        plt.close("all")

        obj, jet, geom, config, initial, shape_params = build_current_setup()
        plot_body_point_cloud(obj)


def on_jet_field_clicked(b):
    with output:
        clear_output(wait=True)
        plt.close("all")

        obj, jet, geom, config, initial, shape_params = build_current_setup()
        sigmas = [max(0.015, jet.sigma*0.5), jet.sigma, min(0.25, jet.sigma*2.0)]
        plot_gaussian_jet_field(jet, geom, sigmas=sigmas)


def on_center_edge_clicked(b):
    with output:
        clear_output(wait=True)
        plt.close("all")

        obj, jet, geom, config, initial, shape_params = build_current_setup()
        run_center_vs_edge_comparison(obj, jet, geom, config, initial)

        print("해석:")
        print("- center hit과 edge hit의 차이는 주로 torque와 angular velocity에서 나타납니다.")
        print("- 최종 성공 판정은 두 경우 모두 x-pass region 기준입니다.")


def on_offset_sweep_clicked(b):
    with output:
        clear_output(wait=True)
        plt.close("all")

        obj, jet, geom, config, initial, shape_params = build_current_setup()
        run_hit_offset_sweep(obj, jet, geom, config, initial)

        print("해석:")
        print("- hit offset은 여기서는 z 방향으로 jet center를 움직이며 분석합니다.")
        print("- x_land가 pass region 안에 들어오는지 여부가 success/fail을 결정합니다.")
        print("- edge-hit에서는 회전 때문에 x_land가 비선형적으로 변할 수 있습니다.")


def on_timing_sweep_clicked(b):
    with output:
        clear_output(wait=True)
        plt.close("all")

        obj, jet, geom, config, initial, shape_params = build_current_setup()
        run_timing_duration_sweep(obj, jet, geom, config, initial)

        print("해석:")
        print("- timing map은 t_on과 duration에 따라 x_land가 pass region에 들어오는지를 보여줍니다.")
        print("- pass-class object라면 success region이 desired operation window입니다.")
        print("- reject-class object라면 pass region 밖으로 떨어지는 조건이 success입니다.")


def on_noise_clicked(b):
    with output:
        clear_output(wait=True)
        plt.close("all")

        obj, jet, geom, config, initial, shape_params = build_current_setup()
        run_noise_trials(obj, jet, geom, config, initial, n_trials=50)

        print("해석:")
        print("- noise가 커지면 x_land distribution이 넓어지고 pass/reject 판정의 repeatability가 떨어질 수 있습니다.")


def on_sorting_clicked(b):
    with output:
        clear_output(wait=True)
        plt.close("all")

        obj, jet, geom, config, initial, shape_params = build_current_setup()
        y_min = min(sort_ymin_w.value, sort_ymax_w.value)
        y_max = max(sort_ymin_w.value, sort_ymax_w.value)

        run_sorting_analysis(
            geom=geom,
            base_jet=jet,
            config=config,
            base_initial=initial,
            current_shape_params=shape_params,
            use_random_y=bool(sort_random_y_w.value),
            y0_min=y_min,
            y0_max=y_max,
            n_per_type=25
        )

        print("해석:")
        print("- sorting analysis에서는 object type별로 pass-class 또는 reject-class를 가정합니다.")
        print("- pass-class는 x-pass region 안에 착지하면 성공입니다.")
        print("- reject-class는 x-pass region 밖에 착지하면 성공입니다.")
        print("- random y0 옵션이 켜져 있으면 넓은 컨베이어 위의 lateral input 위치를 랜덤하게 샘플링합니다.")


def on_sanity_clicked(b):
    with output:
        clear_output(wait=True)
        plt.close("all")

        obj, jet, geom, config, initial, shape_params = build_current_setup()
        run_sanity_checks(obj, jet, geom, config, initial)

        print("Sanity check 해석:")
        print("- gravity-only case가 landing plane에 도달하는지 확인합니다.")
        print("- noise=0이면 반복 결과가 동일해야 합니다.")
        print("- noise>0이면 landing x distribution이 생겨야 합니다.")
        print("- sigma가 커지면 jet 영향을 받는 surface point 수가 증가해야 합니다.")


def on_reset_clicked(b):
    set_baseline()
    update_display_static()


update_btn.on_click(on_update_clicked)
animate_btn.on_click(on_animate_clicked)
object_cloud_btn.on_click(on_object_cloud_clicked)
jet_field_btn.on_click(on_jet_field_clicked)
center_edge_btn.on_click(on_center_edge_clicked)
offset_sweep_btn.on_click(on_offset_sweep_clicked)
timing_sweep_btn.on_click(on_timing_sweep_clicked)
noise_btn.on_click(on_noise_clicked)
sorting_btn.on_click(on_sorting_clicked)
sanity_btn.on_click(on_sanity_clicked)
reset_btn.on_click(on_reset_clicked)


# 11. Dashboard layout


In [ ]:
# ============================================================
# 11. Dashboard layout
# ============================================================

object_box = widgets.VBox([
    widgets.HTML("<h3>Object, class, shape size, and initial orientation</h3>"),
    object_kind_w,
    object_class_w,
    mass_w,
    cd_w,
    surface_resolution_w,
    widgets.HTML("<b>Plate / rod / irregular size parameters</b>"),
    length_x_w,
    width_y_w,
    thickness_z_w,
    widgets.HTML("<b>Cylinder parameters</b>"),
    cyl_radius_w,
    cyl_height_w,
    widgets.HTML("<b>Sphere parameter</b>"),
    sphere_radius_w,
    widgets.HTML("<b>Initial orientation and angular velocity</b>"),
    roll_w,
    pitch_w,
    yaw_w,
    omega_x_w,
    omega_y_w,
    omega_z_w,
])

conveyor_box = widgets.VBox([
    widgets.HTML("""
    <h3>Fixed conveyor and initial COM position</h3>
    <p>
    Conveyor x-range is fixed: <b>x = -0.2 m to x = 0.0 m</b>.
    x is conveyor direction. y is lateral belt coordinate.
    </p>
    """),
    conveyor_width_w,
    conveyor_height_w,
    conveyor_speed_w,
    x0_w,
    y0_w,
])

jet_box = widgets.VBox([
    widgets.HTML("""
    <h3>3D Gaussian air-jet field</h3>
    <p>
    Jet position can be set in x, y, z. 
    azimuth = 0 deg means +x conveyor direction.
    azimuth = 90 deg means +y direction.
    azimuth = -90 deg means -y direction.
    azimuth = 180 deg means -x direction.
    elevation controls vertical z component.
    </p>
    """),
    Umax_w,
    sigma_w,
    ton_w,
    duration_w,
    jet_x_w,
    jet_y_w,
    jet_z_w,
    jet_azimuth_w,
    jet_elevation_w,
    noise_w,
])

sorting_box = widgets.VBox([
    widgets.HTML("""
    <h3>Pass region and numerical settings</h3>
    <p>
    Sorting criterion is based on landing x position.
    pass-class: success if x_land is inside x-pass region.
    reject-class: success if x_land is outside x-pass region.
    </p>
    """),
    Jdetach_w,
    xpass_min_w,
    xpass_max_w,
    tmax_w,
    dt_w,
    seed_w,
    widgets.HTML("<h4>Sorting analysis random-y settings</h4>"),
    sort_random_y_w,
    sort_ymin_w,
    sort_ymax_w,
])

button_box_1 = widgets.HBox([
    update_btn,
    animate_btn,
    object_cloud_btn,
    jet_field_btn,
    reset_btn,
])

button_box_2 = widgets.HBox([
    center_edge_btn,
    offset_sweep_btn,
    timing_sweep_btn,
    noise_btn,
    sorting_btn,
    sanity_btn,
])

controls = widgets.Tab()
controls.children = [object_box, conveyor_box, jet_box, sorting_box]
controls.set_title(0, "Object")
controls.set_title(1, "Conveyor")
controls.set_title(2, "Jet")
controls.set_title(3, "Pass region")

dashboard = widgets.VBox([
    widgets.HTML("""
    <h2>Project 3 Week 2: 3D Rigid-Body Air-Jet Sorting Simulator</h2>
    <h3>Revised x-pass-region sorting criterion</h3>
    <ul>
      <li>x: conveyor direction</li>
      <li>y: lateral belt coordinate and jet targeting coordinate</li>
      <li>z: vertical direction</li>
      <li>Conveyor is fixed from x = -0.2 m to x = 0.0 m.</li>
      <li>Success/fail is determined by landing x relative to the x-pass region.</li>
      <li>Pass-class object succeeds if x_land is inside the x-pass region.</li>
      <li>Reject-class object succeeds if x_land is outside the x-pass region.</li>
      <li>Jet position is adjustable in x, y, z.</li>
      <li>Jet direction is adjustable using azimuth and elevation.</li>
      <li>Object shape includes plate, rod, cylinder, sphere, and irregular flake.</li>
      <li>Object size and surface-point resolution are adjustable.</li>
    </ul>
    """),
    button_box_1,
    button_box_2,
    controls,
    output
])

# Simulation

In [ ]:
display(dashboard)

# Initial result
update_display_static()